# Round-5 — البنود الرخيصة (نقاط تيمون 4، 5، 7أ)

شغّل الخلايا بالترتيب من فوق لتحت.

**مهم:** هاد نوتبوك مستقل — لا توقف نوتبوكات التدريب الشغالة.

- خلية 1: تجهيز (Drive + clone + مكتبات)
- خلية 2: اكتشاف المسارات والتأكد منها **قبل** أي تشغيل
- خلية 3: نقطة 4 — زمن الاستدلال حسب حجم الدفعة (دقائق)
- خلية 4: نقطة 5 — الكميات الفيزيائية: H1، الطاقة، الإجهاد، قوى رد الفعل (~ساعة)
- خلية 5: نقطة 7أ — شبكات أخشن + أنعم
- خلية 6: تجميع النتائج للنسخ واللصق

## خلية 1 — التجهيز

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, sys
os.chdir('/content')
if os.path.exists('/content/OMAR'):
    shutil.rmtree('/content/OMAR')
!git clone -q -b claude/claude-code-question-d307wp https://github.com/suhibamro/omar.git /content/OMAR

WORK = '/content/OMAR/Practical_Examples'
os.chdir(WORK); sys.path.insert(0, WORK)
assert os.path.isdir(os.path.join(WORK, 'omar_pfem')), 'clone فشل'

!{sys.executable} -m pip install -q einops timm h5py jax tqdm
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: فعّل GPU من Runtime > Change runtime type')

## خلية 2 — اكتشاف المسارات (تأكّد إنه كلهم OK قبل ما تكمل)

In [ ]:
import glob, os
R = '/content/drive/MyDrive/pfem_run'
CASES = {
 'B1_neo_hookean':   f'{R}/results/B1_neo_hookean',
 'B1_mooney_rivlin': f'{R}/results/B1_mooney_rivlin',
 'B1_arruda_boyce':  f'{R}/results/B1_arruda_boyce',
 'B2_neo_hookean':   f'{R}/B2_accuracy_search/lossnorm/train',
 'B2_mooney_rivlin': f'{R}/B2_accuracy_search_mooney_rivlin/lossnorm/train',
 'B2_arruda_boyce':  f'{R}/B2_accuracy_search_arruda_boyce/lossnorm/train',
}

# The dataset does not always sit at a fixed offset from the checkpoint --
# the B1 runs keep it somewhere else entirely, which is why the earlier
# fixed-offset globs reported every B1 case as MISSING while the
# checkpoints were sitting right there. Search the tree instead, and pick
# the candidate whose path shares the most with the checkpoint's, so a
# case never silently picks up another case's data.
ALL_NPZ = glob.glob(f'{R}/**/hyperelastic_training_data_q4.npz', recursive=True)
print(f'{len(ALL_NPZ)} dataset files found under {R}\n')

def best_dataset(case_dir, case_name):
    """The dataset belonging to this case, chosen by path, not by guesswork.

    Two rules, in order. First drop any file whose path names a DIFFERENT
    material, so a case can never silently train against another case's
    data (paths for neo_hookean name no material at all, which is why the
    rule excludes rather than requires). Then, among what is left, take
    the file sharing the longest directory prefix with the checkpoint.
    """
    mat = case_name.split('_', 1)[1]
    others = [m for m in ('neo_hookean', 'mooney_rivlin', 'arruda_boyce') if m != mat]
    ok = [p for p in ALL_NPZ if not any(m in p for m in others)]

    parts = case_dir.split('/')
    def shared(p):
        n = 0
        for a, b in zip(parts, p.split('/')):
            if a != b:
                break
            n += 1
        return n

    if not ok:
        return None
    best = max(ok, key=shared)
    return best if shared(best) >= 4 else None

PATHS = {}
for c, d in CASES.items():
    ck = os.path.join(d, 'model_best.pt')
    ds = best_dataset(d, c)
    PATHS[c] = (ck, ds)
    print(f"{c:20s} ckpt={'OK     ' if os.path.exists(ck) else 'MISSING'}  "
          f"data={'OK' if ds else 'MISSING'}")
    if ds: print(f"{'':20s}   -> {ds}")

missing = [c for c,(ck,ds) in PATHS.items() if not (os.path.exists(ck) and ds)]
if missing:
    print(f'\n!! ناقص: {missing}')
    print('كل ملفات البيانات الموجودة، ابعتها لكلود:')
    for p in sorted(ALL_NPZ): print('   ', p)
else:
    print('\nكله تمام — كمّل')

## خلية 3 — نقطة 4: زمن الاستدلال حسب حجم الدفعة
نفس أحجام الدفعات تبع GPU FEM (1/8/32/128) عشان المقارنة تكون عادلة.

In [ ]:
for c, (ck, ds) in PATHS.items():
    if not (os.path.exists(ck) and ds):
        print(f'skip {c}'); continue
    g, m = c.split('_', 1)
    print(f'\n===== {c} =====')
    !python -m omar_pfem.inference_latency_by_batch \
        --geometry {g} --material {m} --checkpoint "{ck}" --dataset "{ds}" \
        --batch_sizes 1,8,32,128 --n_repeats 50 \
        --out_json "{R}/inference_latency_by_batch_{c}.json"

## خلية 4 — نقطة 5: الكميات الفيزيائية
H1، نظيم الطاقة، إجهاد PK1 (كل مركّبة + الذروة)، وقوى رد الفعل.

هاي الأبطأ (~ساعة للستة). لو بدك تجربة سريعة أول، غيّر `--ntest 50` لـ `--ntest 5`.

In [ ]:
for c, (ck, ds) in PATHS.items():
    if not (os.path.exists(ck) and ds):
        print(f'skip {c}'); continue
    g, m = c.split('_', 1)
    print(f'\n===== {c} =====')
    !python -m omar_pfem.physical_quantities_eval \
        --geometry {g} --material {m} --checkpoint "{ck}" --data_path "{ds}" \
        --ntrain 800 --ntest 50 \
        --out_json "{R}/physical_quantities_{c}.json"

## خلية 5 — نقطة 7أ: شبكات أخشن + أنعم
تيمون بدّو الاختبار على شبكات أخشن **و** أنعم من شبكات التدريب (21 و33).
الحالة الجاهزة الوحيدة حاليًا هي B1×Neo-Hookean.

In [ ]:
Z = f'{R}/zeroshot_B1_neo_hookean'
if os.path.exists(f'{Z}/model_best.pt'):
    !python -m omar_pfem.resolution_invariance_zeroshot eval \
        --geometry B1 --material neo_hookean --checkpoint "{Z}/model_best.pt" \
        --test_resolutions 13,17,25,29,37,41,49 --fine_N 101 --n_eval_samples 20 \
        --out_json "{Z}/zeroshot_eval_coarse_and_fine.json"
else:
    print('checkpoint مش موجود:', Z)

## خلية 6 — تجميع النتائج
انسخ كل المخرجات وابعتها لكلود.

In [ ]:
import json, glob
print('='*70)
for pat in ['inference_latency_by_batch_*.json', 'physical_quantities_*.json']:
    for f in sorted(glob.glob(f'{R}/{pat}')):
        print(f'\n----- {os.path.basename(f)} -----')
        print(json.dumps(json.load(open(f)), indent=1)[:4000])

z = f'{R}/zeroshot_B1_neo_hookean/zeroshot_eval_coarse_and_fine.json'
if os.path.exists(z):
    print('\n----- zeroshot coarse+fine -----')
    print(json.dumps(json.load(open(z)), indent=1))
print('\n' + '='*70)